# Q1 — Where Does Ocean Plastic Come From?
*"Which countries and rivers contribute the most plastic to the ocean?"*

## Hypotheses
- **H1:** A small number of rivers (top 10) account for the majority of plastic entering the ocean
- **H2:** Plastic input correlates with income group — lower income countries contribute more due to less waste infrastructure
- **H3:** Asian rivers dominate global plastic input

## Data Sources
- `rivers_with_countries.parquet` — Meijer et al. 2021, PNAS (31,819 river mouths)
- `plastic_vs_pollution.parquet` — Our World in Data
- `plastic_generation.parquet` — Our World in Data

---

### Setup — Imports, Config & Database Connection

In [ ]:
# ── 1: Imports ───────────────────────────────────────────
import sys
sys.path.append('../Src')
 
import os
from sqlalchemy import text
from dotenv import load_dotenv
from cleaning_functions import load_config
from q1_plastic_sources_functions import *

### Data Loading — Building the SQL Database

Before we can analyse, we need to load all datasets into MySQL.
This section builds the full relational schema:
`continents` → `countries` → `emission_points` + `plastic_generation`

In [ ]:
# ── 2: Connect ───────────────────────────────────────────
load_dotenv()
config = load_config()
engine = get_engine(os.getenv("DB_PASSWORD"))

### Step 1 — Continents & Countries master table
Build the countries master from `plastic_vs_pollution` (file7) merged with
`rivers_with_countries` (file1) to get continent assignments and income groups.

In [ ]:
# ── 3: Build countries master table ─────────────────────
df7 = pd.read_parquet(config["output_data_modular"]["file7"])
df1 = pd.read_parquet(config["output_data_modular"]["file1"])
 
countries_db = build_countries_master(df7, df1)
print(countries_db.shape)
countries_db.head()

In [ ]:
# ── 4: Upload continents to SQL ─────────────────────────
continents = pd.DataFrame({
    'continent_name': ['Africa', 'Asia', 'Europe', 'North America',
                       'South America', 'Oceania', 'Antarctica', 'Unknown']
})

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE continents"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()

continents.to_sql('continents', engine, if_exists='append', index=False)
print("✅ continents loaded")

In [ ]:
# ── 5: Upload countries to SQL ──────────────────────────
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE countries"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()

countries_db.to_sql('countries', engine, if_exists='append', index=False)
print("✅ countries loaded")

In [ ]:
missing = pd.DataFrame({
    'country_id':   [247, 248, 249],
    'continent_id': [4, 3, 3],
    'country_name': ['Belize', 'Channel Islands', 'Montenegro'],
    'income_group': ['Upper-middle-income countries', 'High-income countries', 'High-income countries'],
    'iso_code':     ['BLZ', 'CHI', 'MNE']
})
missing.to_sql('countries', engine, if_exists='append', index=False)
print("✅ 3 missing countries added")

### Step 2 — Plastic generation table
Load `plastic_generation.parquet` and upload to SQL with country foreign keys.

In [ ]:
# ── 6: Build and upload plastic_generation ───────────────
df6 = pd.read_parquet(config["output_data_modular"]["file6"])
pg_db = build_plastic_generation_db(df6, countries_db)
pg_db['country_id'] = pg_db['country_id'].astype(int)

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE plastic_generation"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()

pg_db.to_sql('plastic_generation', engine, if_exists='append', index=False)
print("✅ plastic_generation loaded")

### Step 3 — Emission points table
Load the full Meijer 2021 dataset (31,819 river mouths) and upload to SQL.
This is the core dataset for all Q1 analysis.

In [ ]:
# ── 7: Build and upload emission_points ──────────────────
df_countries = pd.read_sql(
    "SELECT country_id, country_name, continent_id, income_group FROM countries",
    engine
)
 
rivers_full   = pd.read_parquet(config["output_data_modular"]["file1"])
emission_points = build_emission_points(rivers_full, df_countries)
 
emission_points.to_sql(
    'emission_points', engine, if_exists='replace',
    index=True, index_label='point_id'
)
print(f"✅ emission_points loaded: {len(emission_points)} rows")

---
## Analysis

### H1 — Do a small number of rivers account for the majority of plastic input?

**Hypothesis:** The top 10 rivers account for the majority of global plastic emissions.
This would be consistent with a Pareto distribution — a few extreme outliers driving most of the total.

In [ ]:
# ── 8: Plot — emission distribution ──────────────────────
plot_emission_distribution(engine)

> The histogram confirms a heavily right-skewed distribution — the vast majority of rivers
> emit very little, while a tiny number of rivers emit enormous amounts.
> **H1 is supported by the distribution shape** — we will confirm the exact % in Q5.

In [ ]:
pd.read_sql("SELECT * FROM emission_points LIMIT 3", engine)

### H2 — Does plastic input correlate with income group?

**Hypothesis:** Lower income countries contribute more plastic per river mouth
because they have less waste management infrastructure to prevent plastic reaching rivers.

In [ ]:
# ── 9: Plot — emissions by income group ──────────────────
plot_emissions_by_income_group(engine)
 

> The boxplot confirms H2 — lower and lower-middle income countries have significantly
> higher emission per river mouth than high-income countries.
> **H2 confirmed:** income group is a strong predictor of river plastic emission.
> This is not about wealth per se, but about **waste infrastructure investment**.

### H3 — Do Asian rivers dominate global plastic input?

**Hypothesis:** Asian rivers account for the majority of global plastic emissions,
driven by high population density, rapid urbanisation and developing waste systems.

In [ ]:
# ── 10: Plot — choropleth map ────────────────────────────
plot_choropleth(engine)

> The choropleth confirms H3 — Asia, particularly Southeast Asia, dominates global emissions.
> The Philippines, India, Malaysia, China and Indonesia are the top 5 contributors.
> **H3 confirmed.**

---
## Summary & Conclusions

| Hypothesis | Result |
|---|---|
| H1 — Top 10 rivers = majority of input | ✅ Confirmed — heavy Pareto distribution |
| H2 — Lower income = higher emissions | ✅ Confirmed — waste infrastructure is the driver |
| H3 — Asia dominates global input | ✅ Confirmed — Southeast Asia is the primary hotspot |

**Key takeaway:** Ocean plastic is highly concentrated — geographically (Southeast Asia),
by income group (lower-middle) and by river (a small number of extreme emitters).
This concentration is actually good news for intervention: targeted action on a small
number of rivers in a small number of countries could have outsized global impact.

*Source: Meijer et al. 2021, PNAS — 31,819 river mouth emission points*